In [1]:
!pip install nltk -q
import nltk
nltk.download('stopwords', quiet=True)

from google.colab import drive
drive.mount('/content/drive')
print("Ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready


In [2]:
import pandas as pd
import numpy as np
import os
import re
import glob
from nltk.corpus import stopwords

BASE_PATH     = "/content/drive/MyDrive/Original Reddit Data"
RAW_PATH      = os.path.join(BASE_PATH, "raw data")
LABELLED_PATH = os.path.join(BASE_PATH, "Labelled Data")
SAVE_PATH     = "/content/drive/MyDrive/"

stop_words = set(stopwords.words('english'))

valid_subreddits = [
    'depression', 'SuicideWatch',
    'mentalhealth', 'Anxiety', 'lonely'
]

def clean_for_vader(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_for_bert(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()
    words = [w for w in text.split() if w not in stop_words]
    return ' '.join(words)

print("Setup complete")
print(f"Years available: {os.listdir(RAW_PATH)}")

Setup complete
Years available: ['2022', '2019', '2021', '2020']


In [3]:
# ── Load and clean Part B ──────────────────────────
import pandas as pd
import os

LABELLED_PATH = "/content/drive/MyDrive/Original Reddit Data/Labelled Data"
SAVE_PATH     = "/content/drive/MyDrive/"

label_files = {
    "Drug and Alcohol" : LABELLED_PATH + "/LD DA 1.csv",
    "Early Life"       : LABELLED_PATH + "/LD EL1.csv",
    "Personality"      : LABELLED_PATH + "/LD PF1.csv",
    "Trauma and Stress": LABELLED_PATH + "/LD TS 1.csv"
}

dfs_b = []
for label, filepath in label_files.items():
    try:
        temp = pd.read_csv(filepath)
        temp['Label'] = label
        dfs_b.append(temp)
        print(f"✓ {label}: {temp.shape[0]} rows")
    except Exception as e:
        print(f"✗ {label}: {e}")

df_b = pd.concat(dfs_b, ignore_index=True)
df_b = df_b.dropna(subset=['selftext'])
if 'CAT 1' in df_b.columns:
    df_b = df_b.drop(columns=['CAT 1'])

df_b['full_text']  = df_b['title'].fillna('') + ' ' + df_b['selftext'].fillna('')
df_b['vader_text'] = df_b['full_text'].apply(clean_for_vader)
df_b['bert_text']  = df_b['full_text'].apply(clean_for_bert)

print(f"\nPart B shape: {df_b.shape}")
print(f"Labels:\n{df_b['Label'].value_counts()}")

✓ Drug and Alcohol: 223 rows
✓ Early Life: 200 rows
✓ Personality: 200 rows
✓ Trauma and Stress: 200 rows

Part B shape: (800, 8)
Labels:
Label
Drug and Alcohol     200
Early Life           200
Personality          200
Trauma and Stress    200
Name: count, dtype: int64


In [ ]:
# ── Save Part B only ───────────────────────────────
SAVE_PATH = "/content/drive/MyDrive/"

df_b.to_csv(SAVE_PATH + "clean_part_b.csv", index=False)
print(f"✓ clean_part_b.csv saved — {df_b.shape[0]} rows")

print("\n=== VERIFICATION ===")
print("clean_part_b.csv:", os.path.exists(SAVE_PATH + "clean_part_b.csv"))
print(f"Labels: {df_b['Label'].value_counts().to_dict()}")
print("Output: clean_part_b.csv — 800 rows")
print("Raw data ready at: Original Reddit Data/raw data/")
print("Notebook 03 will handle Part A processing")

✓ clean_part_b.csv saved — 800 rows

=== VERIFICATION ===
clean_part_b.csv: True
Labels: {'Drug and Alcohol': 200, 'Early Life': 200, 'Personality': 200, 'Trauma and Stress': 200}

Notebook 02 Complete ✅
Output: clean_part_b.csv — 800 rows
Raw data ready at: Original Reddit Data/raw data/
Notebook 03 will handle Part A processing
